In [ ]:
import os
import pathlib
from typing import List
from datasets import load_dataset, concatenate_datasets, ClassLabel
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Config ---
train_lang = 'ru'      
eval_lang = 'ru'    

BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/russian_polish_ipa_12_5_50k/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/russian_polish_normal_12_5_50k/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-merges.txt",
    ),
}

LANG_TO_DATASET = {
    "ru": "iggy12345/sentirueval2016-ipa",
    "pl": "iggy12345/allegro-reviews-ipa"
}

args = {
    'epochs': 8,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_multi_features(examples, features: List[str]) -> List[str]:
    sep = f'\n\n{eod_token}\n\n'
    return [sep.join([x or '' for x in items]) for items in zip(*[examples[f] for f in features])]

def get_fields(example, model_type):
    if model_type == "ipa":
        if 'premise-phoneme' in example:
            return ['premise-phoneme', 'hypothesis-phoneme']
        elif 'sentence_A-phoneme' in example:
            return ['sentence_A-phoneme', 'sentence_B-phoneme']
    if 'premise' in example:
        return ['premise', 'hypothesis']
    return ['sentence_A', 'sentence_B']

def load_and_preprocess(dataset_name, split, tokenizer, model_type):
    ds = load_dataset(dataset_name, split=split, cache_dir=str(args['hf_cache_dir']))
    if 'label' in ds.features and not isinstance(ds.features['label'], ClassLabel):
        ds = ds.cast_column("label", ClassLabel(names=['entailment', 'neutral', 'contradiction']))
    sample = ds[0]
    fields = get_fields(sample, model_type)

    def preprocess(examples):
        features = flatten_multi_features(examples, fields)
        return tokenizer(features, truncation=True, max_length=args['context_size'])

    return ds.map(preprocess, batched=True)

metric = evaluate.load("xnli", "en")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.from_numpy(logits).argmax(dim=-1)
    labels = torch.from_numpy(labels)

    correct = (preds == labels).sum().item()
    total = len(labels)
    accuracy = correct / total

    return {
        "accuracy": accuracy,
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0)
    }

# === Run both IPA and NORMAL models ===
for model_type in ['ipa', 'normal']:
    print(f"\n🔧 Running setup for {model_type.upper()} model")
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)
    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    if train_lang == 'both':
        train_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'train', tokenizer, model_type)
        train_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)
        train_dataset = concatenate_datasets([train_ru, train_pl]).shuffle(seed=42)
    else:
        train_dataset = load_and_preprocess(LANG_TO_DATASET[train_lang], 'train', tokenizer, model_type)

    if eval_lang == 'both':
        eval_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'validation', tokenizer, model_type)
        eval_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)       #change to pl[20%] when required
        eval_dataset = concatenate_datasets([eval_ru, eval_pl])
    elif eval_lang == 'ru':
        eval_dataset = load_and_preprocess(LANG_TO_DATASET['ru'], 'validation', tokenizer, model_type)
    else:
        eval_dataset = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)   #change to pl[20%] when required

    output_dir = pathlib.Path(f"./training_outputs_rupl/{train_lang}2{eval_lang}_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=1,
        metric_for_best_model="precision",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=1000,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        seed=42,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print(f"Training {model_type.upper()} model on {train_lang.upper()} → Evaluating on {eval_lang.upper()}")
    trainer.train()

    print(f"Final evaluation on {eval_lang.upper()} for model {model_type.upper()}")
    results = trainer.evaluate()
    print(results)



🔧 Running setup for IPA model
number of parameters: 123.35M


Map: 100%|██████████| 2490/2490 [00:00<00:00, 8480.30 examples/s]
/tmp/slurmtmp.1601692/ipykernel_3883042/4257188652.py:161: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training IPA model on BOTH → Evaluating on BOTH


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1000,1.295300,0.963782,0.629825,0.579349,0.629825,0.570339
2000,1.010100,0.863388,0.643860,0.653417,0.643860,0.622344
3000,0.940700,0.819539,0.649639,0.641791,0.649639,0.633045


In [3]:
from collections import Counter
from datasets import load_dataset, ClassLabel

# Load dataset
ds = load_dataset("iggy12345/cdsc-e-ipa", split="train")

# Manually cast label column to ClassLabel
label_names = ['entailment', 'neutral', 'contradiction']
ds = ds.cast_column("label", ClassLabel(names=label_names))

# Count label distribution
label_counts = Counter(ds["label"])

print("📊 Class Distribution in Polish Training Set:")
for idx, count in sorted(label_counts.items()):
    print(f"{label_names[idx]:>15}: {count} samples")


Casting the dataset: 100%|██████████| 7200/7200 [00:00<00:00, 336737.87 examples/s]

📊 Class Distribution in Polish Training Set:
     entailment: 559 samples
        neutral: 5344 samples
  contradiction: 1297 samples


In [5]:
import os
import pathlib
from typing import List
from datasets import load_dataset, concatenate_datasets, ClassLabel
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Config ---
BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/russian_polish_ipa_12_5_50k/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/russian_polish_normal_12_5_50k/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-merges.txt",
    ),
}

LANG_TO_DATASET = {
    "ru": "iggy12345/xnli-ru-ipa",
    "pl": "iggy12345/cdsc-e-ipa"
}

args = {
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_multi_features(examples, features: List[str]) -> List[str]:
    sep = f'\n\n{eod_token}\n\n'
    return [sep.join([x or '' for x in items]) for items in zip(*[examples[f] for f in features])]

def get_fields(model_type, dataset_name):
    if model_type == "ipa":
        if "cdsc" in dataset_name:
            return ["sentence_A-phoneme", "sentence_B-phoneme"]
        else:
            return ["premise-phoneme", "hypothesis-phoneme"]
    else:
        if "cdsc" in dataset_name:
            return ["sentence_A", "sentence_B"]
        else:
            return ["premise", "hypothesis"]

def load_and_preprocess(dataset_name, split, tokenizer, model_type):
    ds = load_dataset(dataset_name, split=split, cache_dir=str(args['hf_cache_dir']))
    if 'label' in ds.features and not isinstance(ds.features['label'], ClassLabel):
        ds = ds.cast_column("label", ClassLabel(names=['entailment', 'neutral', 'contradiction']))
    fields = get_fields(model_type, dataset_name)
    def preprocess(examples):
        features = flatten_multi_features(examples, fields)
        return tokenizer(features, truncation=True, max_length=args['context_size'])
    return ds.map(preprocess, batched=True)

metric = evaluate.load("xnli", "en")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.from_numpy(logits).argmax(dim=-1)
    labels = torch.tensor(labels)
    return {
        "accuracy": (preds == labels).sum().item() / len(labels),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# === Train on RU+PL, Evaluate on RU+PL ===
for model_type in ['ipa', 'normal']:
    print(f"Starting run for {model_type.upper()} on BOTH (RU+PL)")

    # Load tokenizer and model
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)
    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # Load + preprocess datasets
    train_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'train', tokenizer, model_type)
    train_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)
    train_dataset = concatenate_datasets([train_ru, train_pl])

    eval_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'validation', tokenizer, model_type)
    eval_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train[:20%]', tokenizer, model_type)
    eval_dataset = concatenate_datasets([eval_ru, eval_pl])

    # Setup training args
    output_dir = pathlib.Path(f"./training_outputs_rupl/both2both_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=1,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=1000,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print(f"Training {model_type.upper()} model on RU+PL → Evaluating on RU+PL")
    trainer.train()

    print(f"Final evaluation on RU+PL for {model_type.upper()}")
    results = trainer.evaluate()
    print(results)


Starting run for IPA on BOTH (RU+PL)
number of parameters: 123.35M


/tmp/slurmtmp.1596623/ipykernel_2800811/3955839931.py:147: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training IPA model on RU+PL → Evaluating on RU+PL


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1000,1.077700,0.997887,0.497455,0.539527,0.497455,0.483182
2000,0.961600,0.917213,0.568957,0.575571,0.568957,0.559686
3000,0.932900,0.874379,0.596692,0.586827,0.596692,0.578650
4000,0.915100,0.894043,0.588295,0.612464,0.588295,0.594469
5000,0.913100,0.851517,0.611705,0.630261,0.611705,0.589040
6000,0.897900,0.866729,0.611450,0.634346,0.611450,0.592221
7000,0.897000,0.878332,0.595420,0.611635,0.595420,0.590366
8000,0.875400,0.829677,0.634606,0.636659,0.634606,0.634896
9000,0.880200,0.794261,0.641985,0.637486,0.641985,0.637202
10000,0.862300,0.793741,0.650382,0.645618,0.650382,0.641682


Final evaluation on RU+PL for IPA


...

/tmp/slurmtmp.1596623/ipykernel_2800811/3955839931.py:147: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training NORMAL model on RU+PL → Evaluating on RU+PL


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1000,1.122900,0.943875,0.547837,0.547054,0.547837,0.542597
2000,0.926700,0.853426,0.628499,0.624841,0.628499,0.622904
3000,0.861600,0.795216,0.666158,0.681121,0.666158,0.645633
4000,0.826400,0.767948,0.663868,0.691267,0.663868,0.668735
5000,0.797100,0.677342,0.719084,0.718915,0.719084,0.712545
6000,0.770000,0.674632,0.714504,0.714743,0.714504,0.707659
7000,0.765500,0.664346,0.724427,0.734369,0.724427,0.721356
8000,0.741900,0.609477,0.736387,0.735909,0.736387,0.732203
9000,0.740400,0.639475,0.739440,0.742242,0.739440,0.734406
10000,0.717400,0.598451,0.746310,0.752703,0.746310,0.739333


Final evaluation on RU+PL for NORMAL


{'eval_loss': 0.5490922927856445, 'eval_accuracy': 0.82264631043257, 'eval_precision': 0.8231742864948823, 'eval_recall': 0.82264631043257, 'eval_f1': 0.8210897804278658, 'eval_runtime': 1.8642, 'eval_samples_per_second': 2108.143, 'eval_steps_per_second': 131.96, 'epoch': 3.0}


In [7]:
import os
import pathlib
from typing import List
from datasets import load_dataset, concatenate_datasets, ClassLabel
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Config ---
BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/russian_polish_ipa_12_5_50k/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/russian_polish_normal_12_5_50k/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-rus-pol-normal-number-preservation-merges.txt",
    ),
}

LANG_TO_DATASET = {
    "ru": "iggy12345/xnli-ru-ipa",
    "pl": "iggy12345/cdsc-e-ipa"
}

args = {
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 32,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_multi_features(examples, features: List[str]) -> List[str]:
    sep = f'\n\n{eod_token}\n\n'
    return [sep.join([x or '' for x in items]) for items in zip(*[examples[f] for f in features])]

def get_fields(model_type, dataset_name):
    if model_type == "ipa":
        if "cdsc" in dataset_name:
            return ["sentence_A-phoneme", "sentence_B-phoneme"]
        else:
            return ["premise-phoneme", "hypothesis-phoneme"]
    else:
        if "cdsc" in dataset_name:
            return ["sentence_A", "sentence_B"]
        else:
            return ["premise", "hypothesis"]

def load_and_preprocess(dataset_name, split, tokenizer, model_type):
    ds = load_dataset(dataset_name, split=split, cache_dir=str(args['hf_cache_dir']))
    if 'label' in ds.features and not isinstance(ds.features['label'], ClassLabel):
        ds = ds.cast_column("label", ClassLabel(names=['entailment', 'neutral', 'contradiction']))
    fields = get_fields(model_type, dataset_name)
    def preprocess(examples):
        features = flatten_multi_features(examples, fields)
        return tokenizer(features, truncation=True, max_length=args['context_size'])
    return ds.map(preprocess, batched=True)

metric = evaluate.load("xnli", "en")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.from_numpy(logits).argmax(dim=-1)
    labels = torch.tensor(labels)
    return {
        "accuracy": (preds == labels).sum().item() / len(labels),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# === Train on RU+PL, Evaluate on RU+PL ===
# === Train on RU+PL, Evaluate on RU and PL separately ===
for model_type in ['ipa', 'normal']:
    print(f"🚀 Starting run for {model_type.upper()} on BOTH (RU+PL)")

    # Load tokenizer and model
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)
    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # Load + preprocess datasets
    train_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'train', tokenizer, model_type)
    train_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train', tokenizer, model_type)
    train_dataset = concatenate_datasets([train_ru, train_pl])

    output_dir = pathlib.Path(f"./training_outputs_rupl/both2split_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=4000,
        save_strategy="steps",
        save_steps=4000,
        save_total_limit=1,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=4000,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=train_ru,  # dummy for init
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print(f"🧠 Training {model_type.upper()} model on RU+PL")
    trainer.train()

    # Evaluate separately on RU
    print(f"\n🔍 Final Evaluation on RU for {model_type.upper()}")
    eval_ru = load_and_preprocess(LANG_TO_DATASET['ru'], 'validation', tokenizer, model_type)
    ru_results = trainer.evaluate(eval_dataset=eval_ru)
    print(ru_results)

    # Evaluate separately on PL
    print(f"\n🔍 Final Evaluation on PL for {model_type.upper()}")
    eval_pl = load_and_preprocess(LANG_TO_DATASET['pl'], 'train[:20%]', tokenizer, model_type)
    pl_results = trainer.evaluate(eval_dataset=eval_pl)
    print(pl_results)


🚀 Starting run for IPA on BOTH (RU+PL)
number of parameters: 123.35M


Map: 100%|██████████| 7200/7200 [00:00<00:00, 10625.57 examples/s]
/tmp/slurmtmp.1596623/ipykernel_2800811/2188855857.py:143: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🧠 Training IPA model on RU+PL


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
4000,0.956200,0.867906,0.602421,0.614892,0.602421,0.601797
8000,0.867300,0.850968,0.614588,0.632266,0.614588,0.614231
12000,0.843900,0.812746,0.636847,0.638631,0.636847,0.636474
16000,0.805100,0.781238,0.656218,0.667442,0.656218,0.655959
20000,0.781800,0.740130,0.677855,0.679835,0.677855,0.678389
24000,0.762100,0.723363,0.689199,0.694787,0.689199,0.689094
28000,0.707700,0.668557,0.716803,0.717551,0.716803,0.716466
32000,0.679200,0.635838,0.733841,0.736961,0.733841,0.734143
36000,0.676200,0.606708,0.749456,0.751345,0.749456,0.749867



🔍 Final Evaluation on RU for IPA


...


🔍 Final Evaluation on RU for NORMAL


Map: 100%|██████████| 2490/2490 [00:00<00:00, 18895.35 examples/s]


{'eval_loss': 0.8224889039993286, 'eval_accuracy': 0.7188755020080321, 'eval_precision': 0.7277596219355167, 'eval_recall': 0.7188755020080321, 'eval_f1': 0.7190978872626289, 'eval_runtime': 0.785, 'eval_samples_per_second': 3171.829, 'eval_steps_per_second': 99.358, 'epoch': 3.0}

🔍 Final Evaluation on PL for NORMAL


Map: 100%|██████████| 1440/1440 [00:00<00:00, 21512.78 examples/s]


{'eval_loss': 0.026300277560949326, 'eval_accuracy': 0.99375, 'eval_precision': 0.9938017955801104, 'eval_recall': 0.99375, 'eval_f1': 0.9937152414049613, 'eval_runtime': 0.4519, 'eval_samples_per_second': 3186.561, 'eval_steps_per_second': 99.58, 'epoch': 3.0}
